# D15 — Construção da Amostra Causal Congelada

Este notebook implementa e audita o contrato causal congelado no D14. Não estima efeitos, ATT, event study, matching ou propensity score. Fonte: `data/processed/cempre_painel_analitico_com_cadastro_causal_2007_2019.parquet`.

## 1. Contrato congelado e pressupostos

A população principal contém 129 tratados candidatos e controles `NEVER_TREATED` do pool estrutural, excluindo integralmente o município com sigilo 708. O painel base deve ser balanceado em 2007–2019; Cabo Frio/2009 é uma exclusão institucional por observação para o estimador futuro.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import plotly.graph_objects as go

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import define_especificacao_causal as d14
import constroi_amostra_causal as amostra_mod
from visualizacao_ipt import CORES_IPT, aplicar_tema_ipt, estilizar_tabela_ipt, salvar_figura_ipt

PAINEL_PATH = ROOT / 'data' / 'processed' / 'cempre_painel_analitico_com_cadastro_causal_2007_2019.parquet'
ARTEFATO_PATH = ROOT / 'data' / 'processed' / 'amostra_causal_cempre_2007_2019.parquet'
EXCLUSOES_PATH = ROOT / 'outputs' / 'diagnostics' / 'exclusoes_amostra_causal.csv'
FIGURES = ROOT / 'outputs' / 'figures'
painel = pd.read_parquet(PAINEL_PATH)
spec = d14.ESPECIFICACAO_CAUSAL_V1
print(f'Entrada offline: {PAINEL_PATH.name} — {len(painel):,} município-ano.')

Entrada offline: cempre_painel_analitico_com_cadastro_causal_2007_2019.parquet — 72,378 município-ano.


In [2]:
contrato = pd.DataFrame([
    ['Tratados', '129 candidatos principais'], ['Controles', 'NEVER_TREATED; pool estrutural menos 5003900'],
    ['Grupo de comparação', spec.grupo_comparacao_principal], ['Período', f'{spec.periodo_total_inicio}–{spec.periodo_total_fim}'],
    ['Outcome', f'{spec.outcome_principal} em nível'], ['Coortes', '2009–2013'],
    ['Antecipação', f'suposição principal = {spec.suposicao_antecipacao_principal_periodos}'],
    ['Cabo Frio', '2009 excluído; g=2010'], ['Sigilo', '5003900 excluído como unidade'],
    ['Balanceamento', 'painel base exigido'], ['Janela principal', 'k=-2…+2'], ['Sensibilidade', 'k=-3…+2'],
], columns=['Item', 'Regra congelada'])
display(estilizar_tabela_ipt(contrato))

,Item,Regra congelada
0,Tratados,129 candidatos principais
1,Controles,NEVER_TREATED; pool estrutural menos 5003900
2,Grupo de comparação,never_treated_pool_estrutural_4964
3,Período,2007–2019
4,Outcome,pessoal_ocupado_assalariado em nível
5,Coortes,2009–2013
6,Antecipação,suposição principal = 0
7,Cabo Frio,2009 excluído; g=2010
8,Sigilo,5003900 excluído como unidade
9,Balanceamento,painel base exigido


## 2. Universo antes dos filtros

In [3]:
universo = pd.DataFrame({
    'medida': ['Municípios', 'Município-ano', 'Fase II', 'Candidatos principais', 'Pool estrutural', 'Nunca expostos'],
    'valor': [painel.codigo_municipio_ibge.nunique(), len(painel), painel.loc[painel.fase_ii == True, 'codigo_municipio_ibge'].nunique(), painel.loc[painel.candidato_amostra_principal == True, 'codigo_municipio_ibge'].nunique(), painel.loc[painel.fl_elegivel_controle_candidato, 'codigo_municipio_ibge'].nunique(), painel.loc[painel.sem_exposicao_observada_2007_2019, 'codigo_municipio_ibge'].nunique()]
})
display(estilizar_tabela_ipt(universo))

,medida,valor
0,Municípios,5570
1,Município-ano,72378
2,Fase II,147
3,Candidatos principais,129
4,Pool estrutural,4964
5,Nunca expostos,4970


## 3. Identidade causal e exclusão do sigilo

A seleção de controles vem exclusivamente do pool institucional congelado; nenhum controle é escolhido por nível, trajetória ou tamanho do outcome.

In [4]:
amostra, exclusoes = amostra_mod.construir_amostra_causal(painel)
codigo = painel.codigo_municipio_ibge.astype(str)
tratados = set(amostra.loc[amostra.papel_causal.eq('TRATADO_PRINCIPAL'), 'codigo_municipio_ibge'])
pool_estrutural = set(codigo[painel.fl_elegivel_controle_candidato])
controles = set(amostra.loc[amostra.papel_causal.eq('CONTROLE_NEVER_TREATED'), 'codigo_municipio_ibge'])
auditoria_identidade = pd.DataFrame([['Tratados principais', len(tratados)], ['Pool estrutural antes do sigilo', len(pool_estrutural)], ['Controle 5003900 excluído', int(d14.CODIGO_MUNICIPIO_SIGILO_EXCLUIR in pool_estrutural)], ['Controles never-treated finais', len(controles)]], columns=['Categoria', 'Unidades'])
display(estilizar_tabela_ipt(auditoria_identidade))
display(estilizar_tabela_ipt(exclusoes))

,Categoria,Unidades
0,Tratados principais,129
1,Pool estrutural antes do sigilo,4964
2,Controle 5003900 excluído,1
3,Controles never-treated finais,4963


,codigo_municipio_ibge,ano,tipo_exclusao,motivo,origem_regra,status_outcome
0,5003900,,UNIDADE,SIGILO_OUTCOME_708_2012,D14,sigilo
1,3300704,2009,OBSERVACAO,ANO_TRANSICAO_INSTITUCIONAL,cadastro_causal_D11_D14,observado


### Interpretação

O município 5003900 é removido inteiro para não imputar nem converter o sigilo de 2012. Cabo Frio permanece tratado: somente 2009 recebe máscara de não elegibilidade, conforme regra institucional já congelada.

## 4. Materialização da população e máscara institucional

In [5]:
# A materialização acima usa a mesma função pura coberta pelos testes D15.
assert amostra.duplicated(['codigo_municipio_ibge','ano']).sum() == 0
assert amostra.codigo_municipio_ibge.nunique() == len(tratados) + len(controles)
assert set(amostra.ano.unique()) == set(range(2007, 2020))
print(f'Unidades: {amostra.codigo_municipio_ibge.nunique():,}; linhas base: {len(amostra):,}; elegíveis ao estimador: {amostra.elegivel_estimacao_principal.sum():,}.')

Unidades: 5,092; linhas base: 66,196; elegíveis ao estimador: 66,195.


## 5. Cabo Frio e balanceamento

O contrato D14 determina um painel causal base balanceado e, separadamente, exclui Cabo Frio/2009 da estimação. A auditoria abaixo não força uma alternativa: verifica que o painel populacional é retangular e que a máscara reduz apenas o conjunto elegível do estimador futuro.

In [6]:
cabo = amostra.loc[amostra.codigo_municipio_ibge.astype(str).eq(d14.CODIGO_CABO_FRIO), ['ano','ano_transicao','primeiro_ano_completo','coorte_g','tempo_relativo','elegivel_estimacao_principal','motivo_nao_elegibilidade_estimacao']]
display(estilizar_tabela_ipt(cabo))
def auditar_painel(frame):
    por_unidade = frame.groupby('codigo_municipio_ibge').ano.nunique()
    return pd.Series({'n_unidades': int(por_unidade.size), 'min_anos': int(por_unidade.min()), 'max_anos': int(por_unidade.max()), 'mediana_anos': float(por_unidade.median()), 'unidades_13_anos': int((por_unidade == 13).sum()), 'unidades_menos_13': int((por_unidade < 13).sum())})
balanceamento_base = amostra.groupby('papel_causal').apply(auditar_painel, include_groups=False).reset_index()
balanceamento_elegivel = amostra.loc[amostra.elegivel_estimacao_principal].groupby('papel_causal').apply(auditar_painel, include_groups=False).reset_index()
display(estilizar_tabela_ipt(balanceamento_base), estilizar_tabela_ipt(balanceamento_elegivel))
alternativas_cabo = pd.DataFrame([['A — máscara por observação', 129, 4963, 5092, 66196, 66195, 'Painel base balanceado; Cabo/2009 não elegível'], ['B — excluir Cabo inteiro', 128, 4963, 5091, 66183, 66183, 'Alternativa diagnóstica não adotada']], columns=['Alternativa','Tratados','Controles','Unidades','Linhas base','Linhas elegíveis','Leitura'])
display(estilizar_tabela_ipt(alternativas_cabo))
CONTRATO_BALANCEAMENTO_REQUER_CLARIFICACAO = 'NAO'
print('Decisão documentada D14: painel base balanceado; máscara institucional por ano no estimador futuro.')

,ano,ano_transicao,primeiro_ano_completo,coorte_g,tempo_relativo,elegivel_estimacao_principal,motivo_nao_elegibilidade_estimacao
41410,2007,2009.000000,2010.000000,2010.000000,-3.000000,True,
41411,2008,2009.000000,2010.000000,2010.000000,-2.000000,True,
41412,2009,2009.000000,2010.000000,2010.000000,-1.000000,False,ANO_TRANSICAO_INSTITUCIONAL_CABO_FRIO_2009
41413,2010,2009.000000,2010.000000,2010.000000,0.000000,True,
41414,2011,2009.000000,2010.000000,2010.000000,1.000000,True,
41415,2012,2009.000000,2010.000000,2010.000000,2.000000,True,
41416,2013,2009.000000,2010.000000,2010.000000,3.000000,True,
41417,2014,2009.000000,2010.000000,2010.000000,4.000000,True,
41418,2015,2009.000000,2010.000000,2010.000000,5.000000,True,
41419,2016,2009.000000,2010.000000,2010.000000,6.000000,True,


,papel_causal,n_unidades,min_anos,max_anos,mediana_anos,unidades_13_anos,unidades_menos_13
0,CONTROLE_NEVER_TREATED,4963.000000,13.000000,13.000000,13.000000,4963.000000,0.000000
1,TRATADO_PRINCIPAL,129.000000,13.000000,13.000000,13.000000,129.000000,0.000000


,papel_causal,n_unidades,min_anos,max_anos,mediana_anos,unidades_13_anos,unidades_menos_13
0,CONTROLE_NEVER_TREATED,4963.000000,13.000000,13.000000,13.000000,4963.000000,0.000000
1,TRATADO_PRINCIPAL,129.000000,12.000000,13.000000,13.000000,128.000000,1.000000


,Alternativa,Tratados,Controles,Unidades,Linhas base,Linhas elegíveis,Leitura
0,A — máscara por observação,129,4963,5092,66196,66195,Painel base balanceado; Cabo/2009 não elegível
1,B — excluir Cabo inteiro,128,4963,5091,66183,66183,Alternativa diagnóstica não adotada


Decisão documentada D14: painel base balanceado; máscara institucional por ano no estimador futuro.


### Decisão sobre balanceamento

Os documentos D14 definem explicitamente que os anos em `anos_excluir_estimacao` são removidos do painel usado pelo estimador para aquele município. Portanto, há três objetos distintos: painel causal base balanceado, outcome observado e subconjunto elegível. A alternativa A implementa o contrato sem excluir Cabo Frio inteiro.

## 6. Fluxo, coortes e janelas

In [7]:
fluxo = pd.DataFrame([['Universo integrado', pd.NA, pd.NA, painel.codigo_municipio_ibge.nunique(), 'Antes dos filtros D15'], ['Candidatos + pool estrutural', len(tratados), len(pool_estrutural), len(tratados) + len(pool_estrutural), 'Pool original de controles'], ['Após exclusão 5003900', len(tratados), len(controles), amostra.codigo_municipio_ibge.nunique(), 'Sigilo excluído como unidade'], ['Painel causal base', len(tratados), len(controles), amostra.codigo_municipio_ibge.nunique(), f'{len(amostra):,} linhas balanceadas'], ['Elegíveis ao estimador futuro', len(tratados), len(controles), amostra.codigo_municipio_ibge.nunique(), f'{int(amostra.elegivel_estimacao_principal.sum()):,} observações; Cabo/2009 excluído']], columns=['Etapa','Tratados','Controles','Total unidades','Observação'])
display(estilizar_tabela_ipt(fluxo))
fig = go.Figure(go.Bar(x=fluxo['Etapa'], y=fluxo['Total unidades'], marker_color=[CORES_IPT['AZUL_CLARO'], CORES_IPT['CIANO'], CORES_IPT['AZUL_PRINCIPAL'], CORES_IPT['AZUL_PRINCIPAL'], CORES_IPT['AZUL_PRINCIPAL']], text=fluxo['Total unidades'], textposition='outside'))
aplicar_tema_ipt(fig, titulo='Construção da amostra causal congelada')
fig.update_yaxes(title='Unidades', rangemode='tozero')
fig.update_xaxes(title='')
salvar_figura_ipt(fig, FIGURES / '06_fluxo_amostra_causal.png')
fig.show()
coortes = amostra.loc[amostra.papel_causal.eq('TRATADO_PRINCIPAL')].groupby('coorte_g').agg(n_tratados=('codigo_municipio_ibge','nunique'), n_linhas_2007_2019=('ano','size'), n_observacoes_elegiveis_painel=('elegivel_estimacao_principal','sum')).reset_index()
coortes['n_observacoes_janela_principal'] = [int(((amostra.tempo_relativo.between(-2,2)) & (amostra.coorte_g.eq(g)) & amostra.elegivel_estimacao_principal).sum()) for g in coortes.coorte_g]
coortes['n_observacoes_sensibilidade'] = [int(((amostra.tempo_relativo.between(-3,2)) & (amostra.coorte_g.eq(g)) & amostra.coorte_g.ge(2010) & amostra.elegivel_estimacao_principal).sum()) for g in coortes.coorte_g]
janela_principal = amostra.loc[amostra.tempo_relativo.between(-2, 2) & amostra.elegivel_estimacao_principal]
janela_sensibilidade = amostra.loc[amostra.tempo_relativo.between(-3, 2) & amostra.coorte_g.ge(2010) & amostra.elegivel_estimacao_principal]
assert janela_principal.codigo_municipio_ibge.nunique() == 129
assert janela_sensibilidade.codigo_municipio_ibge.nunique() == 108
print(f'Janela principal: {janela_principal.codigo_municipio_ibge.nunique()} tratados; sensibilidade: {janela_sensibilidade.codigo_municipio_ibge.nunique()} tratados.')
display(estilizar_tabela_ipt(coortes))

,Etapa,Tratados,Controles,Total unidades,Observação
0,Universo integrado,,,5570,Antes dos filtros D15
1,Candidatos + pool estrutural,129,4964,5093,Pool original de controles
2,Após exclusão 5003900,129,4963,5092,Sigilo excluído como unidade
3,Painel causal base,129,4963,5092,"66,196 linhas balanceadas"
4,Elegíveis ao estimador futuro,129,4963,5092,"66,195 observações; Cabo/2009 excluído"


Janela principal: 129 tratados; sensibilidade: 108 tratados.


,coorte_g,n_tratados,n_linhas_2007_2019,n_observacoes_elegiveis_painel,n_observacoes_janela_principal,n_observacoes_sensibilidade
0,2009.000000,21,273,273,105,0
1,2010.000000,27,351,350,134,161
2,2011.000000,66,858,858,330,396
3,2012.000000,13,169,169,65,78
4,2013.000000,2,26,26,10,12


## 7. Auditorias finais e persistência

In [8]:
auditoria_outcome = amostra.groupby('papel_causal').agg(missing=('pessoal_ocupado_assalariado', lambda s: int(s.isna().sum())), sigilo=('status_pessoal_ocupado_assalariado', lambda s: int((s=='sigilo').sum())), zero=('status_pessoal_ocupado_assalariado', lambda s: int(s.astype(str).str.startswith('zero_').sum())), observado=('status_pessoal_ocupado_assalariado', lambda s: int((s=='observado').sum()))).reset_index()
display(estilizar_tabela_ipt(auditoria_outcome))
assert auditoria_outcome.loc[auditoria_outcome.papel_causal.eq('CONTROLE_NEVER_TREATED'),'sigilo'].iloc[0] == 0
assert len(amostra) == 5092 * 13
assert int(amostra.elegivel_estimacao_principal.sum()) == len(amostra) - 1
amostra.to_parquet(ARTEFATO_PATH, index=False)
exclusoes.to_csv(EXCLUSOES_PATH, index=False, encoding='utf-8')
print(f'Chave única: SIM; artefatos reproduzidos deterministicamente: {ARTEFATO_PATH.name} e {EXCLUSOES_PATH.name}.')

,papel_causal,missing,sigilo,zero,observado
0,CONTROLE_NEVER_TREATED,0,0,3,64516
1,TRATADO_PRINCIPAL,0,0,0,1677


Chave única: SIM; artefato: amostra_causal_cempre_2007_2019.parquet; diagnóstico: exclusoes_amostra_causal.csv.


## 8. Gate D15

`AMOSTRA_CAUSAL_CONSTRUIDA = SIM`

`PRONTO_PARA_IMPLEMENTAR_ESTIMADOR = SIM`

`DESENHO_CAUSAL_APROVADO = NÃO`

A amostra causal está materializada e auditada, mas nenhum efeito foi estimado. A próxima etapa poderá implementar e validar o estimador, sem alterar automaticamente a aprovação do desenho causal.